# Events

This notebook inspects what happens in the model at a detailed step-by-step basis.

Every dollar movement in a run (`transfer_event`), filterable by reason kind, tax character, account and date range. Reads the latest run file for the current scenario under `data/shared/state/runs/`.

The scenario defaults the same way the `sim` CLI resolves it: `$ANALYSIS_SCENARIO` if set, else the name recorded in the data repo's `.current_scenario` (written by `sim load <scenario>`), else `base` — use the `Scenario:` dropdown below to pick a different one, then re-run this cell and the ones below.

The run file is opened read-only via `RunConnection`; `rc.items("transfer_event")` returns one row per money movement, including which rule caused it (`reason_kind`/`rule_name`), its tax character, and the account (and lot, if split) on each end. See [ADR-002](../docs/architecture/ADR-002-storage-schema.md) for the underlying schema.

> **Note:** after running cells, `git status` will show this notebook as modified even with no real changes. Staging it (`git add`) will make it show unmodified again, unless actual changes are present — see [notebooks/README.md](./README.md#git-status-shows-a-notebook-as-modified-after-running-it) for why.

> **Table clipped, or the notebook slow?** Run `analysis jupyter` and open this notebook in JupyterLab, which can use less memory than VS Code.

In [ ]:
%load_ext autoreload
%autoreload 2

from analysis.sim.notebook_ui.pickers import scenario_picker
from analysis.sim.notebook_ui.events_widgets import events_widget

# Pick a different scenario here, then re-run this cell and the ones below.
# CONNECTIONS holds the live ScenarioConnection/RunConnection pair; it's
# refreshed in place whenever the dropdown changes.
PICKER, CONNECTIONS, SCENARIO_PICKER = scenario_picker()
display(PICKER)

## Browse events

- **Accounts**: shows an event if the account is *either* end of the transfer. Leave empty for every account.
- **Reason**: why the event happened — `rule` (a scheduled or one-off rule fired), `refill`/`overflow` (a lot's own balance policy), `ytd`/`ytd_rule` (partial-first-year catch-up facts), `initial` (reserved, not currently emitted). Leave empty for every reason.
- **Tax**: the event's `tax_character`. Leave empty for every character.
- **Date range**: month-end dates; defaults to the whole run.

Leaving a picker empty means "no filter on that axis," not "show nothing."

In [ ]:
CONNECTIONS.show(events_widget)